In [1]:
from astropy.table import Table
from astropy.table import vstack
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import fitsio
import healpy
from scipy.spatial import cKDTree
from astropy.cosmology import Planck18 as cosmo
from mpl_toolkits.mplot3d import Axes3D
import astropy.units as u
import astropy.constants as const
import matplotlib
import astropy.io.ascii
import pandas as pd

In [ ]:
def fibonacciPoints(N):
    ras = []
    decs = []
    for i in np.arange(-N, N+1):
        ra = np.arcsin(2*i/(2*N+1))
        dec = np.mod(i,ra)*360/ra
        ra = ra*180/np.pi
        if dec < 0:
            dec += 360
        ras.append(ra)
        decs.append(dec)
    return [ras,decs]

def raDecToUnitSphere(a, d):
    a = a*np.pi/180
    d = d*np.pi/180
    x = np.cos(a)*np.cos(d)
    y = np.sin(a)*np.cos(d)
    z = np.sin(d)
    return [np.array([x[i],y[i],z[i]]) for i in range(len(a))]

def checkForHoles(randtable,holesep,holerad):
    holearearad = holesep**2*(np.pi/180)**2
    nholes = 4*np.pi/holearearad
    N = nholes/2
    holecenters = fibonacciPoints(N)
    randtable["xyz"] = raDecToUnitSphere(randtable['RA'],randtable['DEC'])
    randmap = [coord for coord in randtable['xyz']]
    holemap = raDecToUnitSphere(holecenters)
    randsphere = ckDTree(randmap)
    neighbors = randsphere.query_ball_point(holemap,holerad)
    holemask = [(len(n) >= 1) for n in neighbors]